In [ ]:
import pathlib
import random
import copy
import numpy as np
import torch

from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

from sliced_wasserstein import sliced_wasserstein_distance
from c2st import c2st_knn, c2st_nn, c2st_rf

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
import seaborn as sns

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""

def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))

In [ ]:
def load_distances(subject_index, rep=1):
    dir = "/home/marco/Documents/GitHub/tms_eeg_decoding/perturb_samples_power/parallel_perturbation_distance"
    
    distances = {}

    for band_name, (low_freq, high_freq) in freq_bands.items():
        distances[band_name] = {}

        for factor in amplification_factors:
            file_path = f"parallel_distance_dict_{band_name}_channel_amp_factor_{factor}_subject_{subject_index}_rep_{rep}.npy"
            load_path = os.path.join(dir, file_path)
            distances[band_name][factor] = np.load(load_path, allow_pickle=True).item()

    return distances

In [ ]:
cfg = load_config()

In [ ]:
freq_bands = {
              "delta": (0, 4),
              "theta": (0, 4),             
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}

amplification_factors = [0.5,0.8,0.9,1.1,1.2,1.5]

In [ ]:
all_subjects_distances = {}
for subject_index in cfg.dataset.test_subject_indices:
    print(f"Loading subject {subject_index}")
    distances = load_distances(subject_index)
    all_subjects_distances[subject_index] = distances

In [ ]:
all_subjects_distances[1]["theta"][1.1].keys()

In [ ]:
def plot_distances_histogram(freq_band, amplification_factor, figsize=(10, 4), bins=30):
    """
    Plot a histogram of distances for all subjects for a given frequency band and amplification factor
    using seaborn for enhanced visualization.
    
    Parameters:
    ----------
    freq_band : str
        The frequency band (e.g., 'alpha', 'beta', etc.)
    amplification_factor : float
        The amplification factor (e.g., 0.5, 0.8, etc.)
    figsize : tuple
        Figure size for the plot
    bins : int
        Number of bins for the histogram
    """
    # Collect distances from all subjects
    all_distances = []
    
    for subject_idx in all_subjects_distances.keys():
        # Get the distance dictionary for this subject, freq band, and amplification factor
        distance_dict = all_subjects_distances[subject_idx][freq_band][amplification_factor]
        
        # Extract values from the dictionary
        if isinstance(distance_dict, dict):
            values = list(distance_dict.values())
            # Flatten if necessary
            for val in values:
                if isinstance(val, (list, np.ndarray)):
                    all_distances.extend(val)
                elif isinstance(val, (float, int)):
                    all_distances.append(val)
        elif isinstance(distance_dict, (list, np.ndarray)):
            all_distances.extend(distance_dict)
    
    # Create the histogram with seaborn
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim([0,100])
    sns.set_style("whitegrid")
    ax = sns.histplot(all_distances, bins="auto", kde=True, color="skyblue", edgecolor="black", alpha=0.7, ax=ax)
    ax.set_xlim([0,100])  # Set x-axis limit to start from 0
    
    # Add labels and title
    plt.xlabel('Distance', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title(f'Distribution of Distances for {freq_band.capitalize()} Band (Amplification Factor: {amplification_factor})', 
              fontsize=14)
    
    # Add statistics
    mean_val = np.mean(all_distances)
    median_val = np.median(all_distances)
    
    plt.axvline(mean_val, color='red', linestyle='dashed', linewidth=1, 
                label=f'Mean: {mean_val:.4f}')
    plt.axvline(median_val, color='green', linestyle='dashed', linewidth=1,
                label=f'Median: {median_val:.4f}')
    
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    return all_distances

In [ ]:
def plot_all_frequency_bands(amplification_factor, figsize=(15, 12)):
    """
    Plot histograms of distances for all frequency bands for a given amplification factor 
    in a single figure with subplots.
    
    Parameters:
    ----------
    amplification_factor : float
        The amplification factor (e.g., 0.5, 0.8, etc.)
    figsize : tuple
        Figure size for the plot
    """
    # Determine a reasonable layout
    n_freq_bands = len(freq_bands)
    nrows = 3  # 3x2 layout for 5 frequency bands
    ncols = 2
    
    # Set up the figure with subplots
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=figsize)
    
    # Flatten axes for easier indexing
    axes = axes.flatten()
    
    # Set the style
    sns.set_style("whitegrid")
    
    # Create a dictionary to store all distances for later return
    all_freq_distances = {}
    
    # Loop through each frequency band and create a subplot
    for i, (freq_band, _) in enumerate(freq_bands.items()):
        if i >= len(axes):  # Safety check
            break
            
        # Get the current axis
        ax = axes[i]
        
        # Collect distances from all subjects for this frequency band
        all_distances = []
        
        for subject_idx in all_subjects_distances.keys():
            # Get the distance dictionary for this subject, freq band, and amplification factor
            distance_dict = all_subjects_distances[subject_idx][freq_band][amplification_factor]
            
            # Extract values from the dictionary
            if isinstance(distance_dict, dict):
                values = list(distance_dict.values())
                # Flatten if necessary
                for val in values:
                    if isinstance(val, (list, np.ndarray)):
                        all_distances.extend(val)
                    elif isinstance(val, (float, int)):
                        all_distances.append(val)
            elif isinstance(distance_dict, (list, np.ndarray)):
                all_distances.extend(distance_dict)
        
        # Store distances for return
        all_freq_distances[freq_band] = all_distances
        
        # Plot histogram for this frequency band
        ax.set_xlim([0, 200])
        sns.histplot(all_distances, bins="auto", kde=True, color="skyblue", 
                     edgecolor="black", alpha=0.7, ax=ax)
        
        # Add labels and title for this subplot
        ax.set_xlabel('Distance', fontsize=10)
        ax.set_ylabel('Frequency', fontsize=10)
        ax.set_title(f'{freq_band.capitalize()} Band', fontsize=12)
        
        # Add statistics
        mean_val = np.mean(all_distances)
        median_val = np.median(all_distances)
        
        ax.axvline(mean_val, color='red', linestyle='dashed', linewidth=1, 
                  label=f'Mean: {mean_val:.4f}')
        ax.axvline(median_val, color='green', linestyle='dashed', linewidth=1,
                  label=f'Median: {median_val:.4f}')
        
        ax.legend(fontsize=8)
    
    # Hide any unused subplots
    for j in range(n_freq_bands, len(axes)):
        axes[j].axis('off')
    
    # Add a main title for the entire figure
    fig.suptitle(f'Distribution of Distances for All Frequency Bands (Amplification Factor: {amplification_factor})', 
                fontsize=16)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Leave space for the suptitle
    plt.show()
    
    return all_freq_distances

In [ ]:
plot_all_frequency_bands(1.1, figsize=(15, 12))